# Transfer CoAtNet labels to normalized training images

Reads **gany24558/rsna-knee-normalized-all-training-data**, predicts 12 study-level soft labels with the released **0.949 CoAtNet checkpoint**, and uploads **gany24558/normalized-images-transferred-labels**. The output includes the original normalized image archives, not just references.

Run on **GPU T4**, with **Internet enabled** for dependencies, input attachment if needed, and publication. No competition test images are read. This notebook is self-contained; support files need not be uploaded.

## Input adaptation
The teacher's published score uses its own raw-DICOM preprocessing. Our cache has already been intensity-normalized, spatially aligned, resampled and slice-subsampled. The adapter restores acquisition axes using stored geometry, selects the teacher's five sequence slots (26/22/18/12/18 slices), makes a 140mm bounded crop at 384 pixels, and takes the central 320 pixels. It averages plain and anatomical-mirror **sigmoid probabilities**, without percentile ranking or hard thresholds. This adaptation cannot recover clipped intensities or discarded detail, so **0.949 is not a claimed score for this adapted pipeline**.

The teacher's predictions are not OOF; original gold/report labels and folds are retained separately. Its documented OAI provenance is recorded in the output; competition eligibility remains unverified.

Teacher: [Danial Zakaria's notebook](https://www.kaggle.com/code/nartaa/rsna-knee-0949-anatomical-mirror), version 2. Architecture credited to dreaddevelopment's Raptor CoAtNet/MIL. Adapted model source retains the original Apache-2.0 attribution.


In [1]:
from pathlib import Path
NORMALIZED_ROOT = Path('/kaggle/input/datasets/gany24558/rsna-knee-normalized-all-training-data')
WEIGHTS_ROOT = Path('/kaggle/input/datasets/nartaa/rsna-knee-publication-swa-weights-20261007')
WORK_DIR = Path('/kaggle/working/coatnet-label-transfer-progress')
OUTPUT_DIR = Path('/kaggle/working/normalized-images-transferred-labels')
PUBLISH_HANDLE = 'gany24558/normalized-images-transferred-labels'
MAX_HOURS = 7.5
# For a new Kaggle session, attach the prior notebook output and set its progress folder here.
RESUME_ROOT = None


## 1. Prepare dependencies and embedded runtime
The teacher release records timm 1.0.29. This cell installs that version only when needed. KaggleHub uses the authenticated Kaggle notebook session; no credentials are embedded.

In [2]:
import importlib.metadata, subprocess, sys
try: timm_version = importlib.metadata.version('timm')
except importlib.metadata.PackageNotFoundError: timm_version = None
if timm_version != '1.0.29':
    if 'timm' in sys.modules: raise RuntimeError('Restart the kernel before switching timm versions.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'timm==1.0.29'])
try: import kagglehub
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'kagglehub'])
    import kagglehub


In [3]:
import sys, importlib
SOURCE_FILES = {'runtime.py': '"""Transfer the released CoAtNet probabilities to the native-plane normalized cache."""\nimport hashlib, io, json, os, shutil, tarfile, time\nfrom pathlib import Path, PurePosixPath\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.nn import functional as F\n\nID, SID = \'StudyInstanceUID\', \'SeriesInstanceUID\'\nTARGETS = [\'ACL\',\'MCL\',\'Medial Meniscus\',\'Lateral Meniscus\',\'Medial OA\',\'Lateral OA\',\n           \'PF OA\',\'Effusion\',\'Synovitis\',"Baker\'s",\'Contusion\',\'Fracture\']\nSOURCE = \'gany24558/rsna-knee-normalized-all-training-data\'\nTEACHER = \'nartaa/rsna-knee-publication-swa-weights-20261007/versions/1\'\nWEIGHT = \'raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt\'\nSHA = \'7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef\'\nSLOTS = [(\'Sagittal\',1,26),(\'Sagittal\',0,22),(\'Coronal\',1,18),(\'Coronal\',0,12),(\'Axial\',-1,18)]\n\ndef digest(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda:f.read(8<<20), b\'\'): h.update(b)\n    return h.hexdigest()\n\ndef save_json(path, obj):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_suffix(path.suffix+\'.tmp\')\n    tmp.write_text(json.dumps(obj,indent=2,allow_nan=False,default=str));tmp.replace(path)\n\ndef safe(arc,name):\n    p=PurePosixPath(name)\n    if p.is_absolute() or \'..\' in p.parts: raise ValueError(\'Unsafe archive path\')\n    m=arc.getmember(name)\n    if not m.isfile(): raise ValueError(\'Not a regular archive file\')\n    return m\n\ndef inspect_source(root):\n    root=Path(root); index=json.loads((root/\'dataset_index.json\').read_text())\n    if index.get(\'dataset_handle\')!=SOURCE: raise ValueError(\'Wrong source dataset handle\')\n    ident=index[\'identity\']; frames=[]; prep=None; labels=None\n    if set(index[\'shards\'])!={f\'{i:03d}\' for i in range(int(ident[\'num_shards\']))}:\n        raise ValueError(\'Missing source shards\')\n    for slot,entry in sorted(index[\'shards\'].items()):\n        name=entry[\'file\']\n        if Path(name).name!=name or entry[\'status\']!=\'processed\': raise ValueError(\'Invalid/partial shard\')\n        with tarfile.open(root/name,\'r:\') as arc:\n            pub=json.load(arc.extractfile(safe(arc,\'publication_manifest.json\')))\n            if pub[\'status\']!=\'processed\' or any(pub.get(k)!=v for k,v in ident.items()):\n                raise ValueError(\'Source identity mismatch\')\n            cfg=json.load(arc.extractfile(safe(arc,\'preprocessing_config.json\')))\n            if cfg[\'run_id\']!=ident[\'run_id\'] or (prep is not None and cfg!=prep):\n                raise ValueError(\'Preprocessing mismatch\')\n            prep=cfg\n            raw=arc.extractfile(safe(arc,\'study_labels_and_folds.csv\')).read()\n            if hashlib.sha256(raw).hexdigest()!=ident[\'label_sha256\']: raise ValueError(\'Label checksum mismatch\')\n            if labels is None: labels=pd.read_csv(io.BytesIO(raw),dtype={ID:str})\n            f=pd.read_csv(arc.extractfile(safe(arc,f\'series_manifest_shard_{int(slot):03d}.csv\')),\n                          dtype={ID:str,SID:str})\n            f[\'source_package\']=name\n            offsets=[];sizes=[]\n            for r in f.to_dict(\'records\'):\n                if r[\'status\']==\'ok\':\n                    m=safe(arc,r[\'cache_path\']);offsets.append(m.offset_data);sizes.append(m.size)\n                else: offsets.append(0);sizes.append(0)\n            f[\'offset\']=offsets;f[\'nbytes\']=sizes;frames.append(f)\n    series=pd.concat(frames,ignore_index=True)\n    if series.empty or series.duplicated([ID,SID]).any(): raise ValueError(\'Empty/duplicate series\')\n    required={ID,SID,\'Anatomical_Plane\',\'Fluid_Sensitive\',\'cache_path\',\'status\'}\n    if not required.issubset(series): raise ValueError(\'Missing sequence metadata\')\n    if not labels[ID].is_unique: raise ValueError(\'Duplicate source study labels\')\n    expected=set(series[ID]); good=series[series.status.eq(\'ok\')]\n    if expected!=set(good[ID]): raise ValueError(\'Some studies have no usable cached series\')\n    if not expected.issubset(set(labels[ID])): raise ValueError(\'Studies lack source label metadata\')\n    return series,labels,index,prep\n\ndef read_cache(root,row,run_id):\n    with (Path(root)/row[\'source_package\']).open(\'rb\') as f:\n        f.seek(int(row[\'offset\']));raw=f.read(int(row[\'nbytes\']))\n    if len(raw)!=int(row[\'nbytes\']): raise ValueError(\'Truncated cache\')\n    with np.load(io.BytesIO(raw),allow_pickle=False) as z:\n        image=z[\'image\'];meta=json.loads(str(z[\'meta\'].item()))\n        support=np.unpackbits(z[\'valid_bits\'],count=image.size).reshape(image.shape).astype(bool)\n    if image.shape!=(64,320,320) or image.dtype!=np.float16: raise ValueError(\'Wrong image contract\')\n    if meta.get(\'representation\')!=\'native-plane-full-fov-v3\' or meta.get(\'run_id\')!=run_id:\n        raise ValueError(\'Wrong cache representation\')\n    if any(meta.get(k)!=row[k] for k in (ID,SID)): raise ValueError(\'Cache ID mismatch\')\n    n=int(meta[\'selected_count\'])\n    if not 1<=n<=64 or support[n:].any() or not support[:n].reshape(n,-1).any(1).all():\n        raise ValueError(\'Invalid support mask\')\n    if not np.isfinite(image).all() or image.min()<0 or image.max()>1: raise ValueError(\'Invalid pixels\')\n    positions=np.asarray(meta[\'slice_positions_mm\'])\n    if len(positions)!=n or not np.isfinite(positions).all() or (n>1 and (np.diff(positions)<=0).any()):\n        raise ValueError(\'Invalid physical slice ordering\')\n    return image[:n],support[:n],meta\n\ndef native_teacher_slices(image,support,meta,k):\n    """Restore acquisition axes from geometry, sample 2--98%, crop 140mm ->384->320.\n    The original intensity distribution and discarded slices cannot be recovered.\n    """\n    b=np.asarray(meta[\'input_basis_lps\'],float)\n    out=np.asarray(meta[\'output_inplane_basis_lps\'],float)\n    shape=np.asarray(meta[\'input_shape\'],int)\n    if b.shape!=(3,3) or out.shape!=(3,2) or shape.shape!=(3,): raise ValueError(\'Missing geometry\')\n    if not np.isfinite(b).all() or not np.isfinite(out).all() or np.linalg.matrix_rank(out)!=2:\n        raise ValueError(\'Invalid geometry\')\n    # DICOM normal = cross(column-index direction, row-index direction).\n    normal=np.cross(b[:,2],b[:,1]); normal/=np.linalg.norm(normal)\n    if np.dot(b[:,0],normal)<0: image=image[::-1];support=support[::-1]\n    n=len(image);lo=int(n*.02);hi=max(lo,int(n*.98)-1)\n    picks=np.linspace(lo,hi,k).round().astype(int).clip(0,n-1)\n    x=torch.from_numpy(np.ascontiguousarray(np.where(support[picks],image[picks],0))).float()[:,None]\n    # Source reader uses one scalar PixelSpacing and a square crop bounded by source size.\n    ps=float(np.linalg.norm(b[:,1])); crop_px=min(int(round(140/max(ps,1e-3))),int(min(shape[1:])))\n    if crop_px<1 or ps<=0: raise ValueError(\'Invalid physical extent\')\n    axes=b[:,1:]/np.linalg.norm(b[:,1:],axis=0)\n    mapping=np.linalg.pinv(out)@axes\n    # Pixel centers for OpenCV-style resize of crop_px source pixels to 384 pixels.\n    v=(np.arange(384)+.5)*(crop_px*ps/384)-crop_px*ps/2\n    rr,cc=np.meshgrid(v,v,indexing=\'ij\')\n    coords=np.stack([rr,cc],-1)@mapping.T + (np.array(image.shape[1:])-1)/2\n    grid=np.stack([2*coords[...,1]/319-1,2*coords[...,0]/319-1],-1).astype(\'float32\')\n    y=F.grid_sample(x,torch.from_numpy(grid)[None].expand(k,-1,-1,-1),\n                    mode=\'bilinear\',padding_mode=\'zeros\',align_corners=True)[:,0]\n    # Match teacher\'s uint8 quantization before its central 320px crop.\n    return (y.clamp(0,1)*255).to(torch.uint8).numpy()\n\ndef build_volume(root,rows,run_id):\n    vol=np.zeros((96,384,384),np.uint8);used=set();audit=[];offset=0\n    for plane,fluid,k in SLOTS:\n        c=[r for r in rows if r[\'status\']==\'ok\' and r[\'Anatomical_Plane\']==plane and r[SID] not in used]\n        pref=[r for r in c if pd.notna(r[\'Fluid_Sensitive\']) and int(r[\'Fluid_Sensitive\'])==fluid]\n        chosen=(pref or c) if fluid in (0,1) else c\n        if chosen:\n            r=chosen[0];used.add(r[SID]);im,sup,meta=read_cache(root,r,run_id)\n            vol[offset:offset+k]=native_teacher_slices(im,sup,meta,k)\n            audit.append({\'plane\':plane,\'fluid_preference\':fluid,SID:r[SID],\'cached_slices\':len(im)})\n        offset+=k\n    mask=(vol.reshape(96,-1).sum(1)>0)\n    if not mask.any(): raise ValueError(\'No nonzero teacher inputs\')\n    return vol,mask,audit\n\ndef windows(vol,mask,mirror=False):\n    valid=np.flatnonzero(mask);lo=int(valid.min());hi=int(valid.max())\n    centers=list(range(lo+1,hi)) or [max(1,min((lo+hi)//2,94))]\n    cs=np.asarray(centers)[np.linspace(0,len(centers)-1,94).round().astype(int)]\n    tri=vol[np.stack([cs-1,cs,cs+1],1)]\n    if mirror:\n        sag=cs<48\n        tri=np.where(sag[:,None,None,None],tri[:,::-1],tri[:,:,:,::-1])\n    x=torch.from_numpy(np.ascontiguousarray(tri[:,:,32:352,32:352])).float()/255\n    return (x-torch.tensor([.485,.456,.406])[None,:,None,None])/torch.tensor([.229,.224,.225])[None,:,None,None]\n\n@torch.inference_mode()\ndef predict(model,vol,mask,device):\n    probabilities=[]\n    for mirror in (False,True):\n        x=windows(vol,mask,mirror)[None].to(device)\n        with torch.autocast(\'cuda\',dtype=torch.float16): logits=model(x).float()\n        if logits.shape!=(1,12) or not torch.isfinite(logits).all(): raise ValueError(\'Invalid teacher logits\')\n        probabilities.append(logits.sigmoid()[0].cpu().numpy())\n        del x\n    return np.mean(probabilities,axis=0)\n\ndef run(root,weights,work,max_hours=7.5,predictor=None,model=None):\n    root,work=Path(root),Path(work);work.mkdir(parents=True,exist_ok=True)\n    series,labels,index,prep=inspect_source(root)\n    if digest(weights)!=SHA: raise ValueError(\'Teacher checkpoint hash mismatch\')\n    identity={\'source_index_sha256\':digest(root/\'dataset_index.json\'),\'teacher_sha256\':SHA,\n              \'runtime_sha256\':digest(__file__), \'teacher_model_sha256\':digest(Path(__file__).with_name(\'teacher_model.py\')),\n              \'recipe\':\'normalized-to-raptor-v1;raw-sigmoid;plain-mirror-mean\'}\n    stamp=work/\'run_identity.json\'\n    if stamp.exists() and json.loads(stamp.read_text())!=identity: raise ValueError(\'Resume identity mismatch; use new WORK_DIR\')\n    save_json(stamp,identity)\n    if predictor is None:\n        if not torch.cuda.is_available(): raise RuntimeError(\'Enable a CUDA GPU in Kaggle\')\n        from teacher_model import load_model\n        model,_=load_model(weights,\'coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k\',384,\'cuda:0\')\n        predictor=lambda v,m:predict(model,v,m,\'cuda:0\')\n    checkpoint=work/\'predictions\';checkpoint.mkdir(exist_ok=True);start=time.monotonic();results=[];audits=[]\n    for sid,group in series.groupby(ID,sort=True):\n        key=hashlib.sha256(sid.encode()).hexdigest();p=checkpoint/(key+\'.json\')\n        if p.exists():\n            item=json.loads(p.read_text())\n            if item[ID]!=sid or item[\'identity\']!=identity: raise ValueError(\'Prediction resume mismatch\')\n        else:\n            if time.monotonic()-start>max_hours*3600:\n                raise RuntimeError(\'Time budget reached. Save notebook output, attach it, restore WORK_DIR and resume; nothing published.\')\n            vol,mask,audit=build_volume(root,group.to_dict(\'records\'),index[\'identity\'][\'run_id\'])\n            probs=np.asarray(predictor(vol,mask),float)\n            item={ID:sid,\'identity\':identity,\'probabilities\':probs.tolist(),\'slots\':audit}\n            validate_probs(probs);save_json(p,item)\n        validate_probs(np.asarray(item[\'probabilities\']))\n        results.append({ID:sid,**dict(zip(TARGETS,item[\'probabilities\']))})\n        audits.append({ID:sid,\'slots\':item[\'slots\']})\n        if len(results)%25==0: print(\'Labelled\',len(results),\'/\',series[ID].nunique(),flush=True)\n    frame=pd.DataFrame(results)\n    if set(frame[ID])!=set(series[ID]) or not frame[ID].is_unique: raise ValueError(\'Incomplete prediction coverage\')\n    tmp=work/\'transferred_labels.csv.tmp\';frame.to_csv(tmp,index=False);tmp.replace(work/\'transferred_labels.csv\')\n    save_json(work/\'slot_audit.json\',audits)\n    save_json(work/\'complete.json\',dict(identity=identity,studies=len(frame),series=len(series)))\n    return frame,series,labels,index,prep\n\ndef validate_probs(p):\n    if p.shape!=(12,) or not np.isfinite(p).all() or (p<0).any() or (p>1).any():\n        raise ValueError(\'Expected twelve finite probabilities in [0,1]\')\n\ndef package(root,work,output,frame,series,labels,index,prep,handle):\n    root,work,output=map(Path,(root,work,output))\n    if output.exists(): raise FileExistsError(\'Output already exists; use it to retry upload or choose a new OUTPUT_DIR\')\n    complete=json.loads((work/\'complete.json\').read_text())\n    if complete[\'identity\'][\'source_index_sha256\']!=digest(root/\'dataset_index.json\'): raise ValueError(\'Source changed\')\n    if not frame.equals(pd.read_csv(work/\'transferred_labels.csv\',dtype={ID:str})):\n        # CSV precision round trips are compared numerically, with exact IDs and columns.\n        disk=pd.read_csv(work/\'transferred_labels.csv\',dtype={ID:str})\n        if list(frame.columns)!=list(disk.columns) or frame[ID].tolist()!=disk[ID].tolist() or not np.allclose(frame[TARGETS],disk[TARGETS],rtol=0,atol=1e-12):\n            raise ValueError(\'Transferred labels changed\')\n    for p in frame[TARGETS].to_numpy():validate_probs(p)\n    if len(frame)!=complete[\'studies\'] or set(frame[ID])!=set(series[ID]): raise ValueError(\'Coverage mismatch\')\n    stage=output.with_name(output.name+\'.building\');stage.mkdir(parents=True,exist_ok=True)\n    files=sorted({e[\'file\'] for e in index[\'shards\'].values()});inventory=[]\n    for name in files:\n        source=root/name;target=stage/name\n        if not target.exists():\n            if shutil.disk_usage(stage).free<source.stat().st_size+2e9: raise OSError(\'Not enough disk to copy image archives; keep WORK_DIR and retry with more space\')\n            temp=target.with_suffix(target.suffix+\'.tmp\');shutil.copyfile(source,temp);temp.replace(target)\n        if source.stat().st_size!=target.stat().st_size or digest(source)!=digest(target): raise ValueError(\'Image archive copy mismatch\')\n        inventory.append({\'file\':name,\'bytes\':target.stat().st_size,\'sha256\':digest(target)})\n    for name in [\'transferred_labels.csv\',\'slot_audit.json\']:shutil.copyfile(work/name,stage/name)\n    labels.to_csv(stage/\'original_study_labels_and_folds.csv\',index=False)\n    portable=series.drop(columns=[\'offset\',\'nbytes\']);portable.to_csv(stage/\'series_manifest.csv\',index=False)\n    save_json(stage/\'source_dataset_index.json\',index)\n    save_json(stage/\'preprocessing_config.json\',prep)\n    manifest=dict(schema_version=1,dataset_handle=handle,source_dataset=SOURCE,\n        teacher_dataset=TEACHER,teacher_notebook=\'nartaa/rsna-knee-0949-anatomical-mirror v2\',\n        **complete,targets=TARGETS,label_file=\'transferred_labels.csv\',label_type=\'soft probabilities\',\n        label_sha256=digest(stage/\'transferred_labels.csv\'),series_manifest_sha256=digest(stage/\'series_manifest.csv\'),\n        image_archives=inventory,preprocessing_exact_match=False,teacher_predictions_are_oof=False,\n        training_provenance=\'Released teacher acknowledges OAI-derived training resources; competition eligibility not verified.\')\n    save_json(stage/\'transfer_manifest.json\',manifest)\n    (stage/\'README.md\').write_text(\'\'\'# Normalized images with transferred CoAtNet labels\n\nUse transferred_labels.csv as the authoritative teacher target table (12 soft probabilities).\nOriginal expert/report labels and folds are preserved separately; teacher predictions are not OOF.\nAll original image archives are included byte-for-byte. Read source_dataset_index.json for their\nlayout and series_manifest.csv for cache paths within source_package. The labels inside those\narchives are ORIGINAL labels, not transferred labels. Do not use the old training loader unchanged:\njoin transferred_labels.csv by StudyInstanceUID and use soft-target BCE. Do not threshold or rank.\n\nThis is adapted inference on normalized native-plane images, not a reproduction of 0.949.\nCached intensity clipping, spatial resampling and discarded slices cannot be undone.\n\nTeacher: Danial Zakaria / nartaa, RSNA Knee 0949 Anatomical Mirror v2 (Apache-2.0 notebook).\nArchitecture: Raptor CoAtNet/MIL, credited to dreaddevelopment. Backbone from timm.\nSource: gany24558/rsna-knee-normalized-all-training-data.\nTeacher checkpoint SHA-256: 7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef.\nTeacher acknowledges OAI-derived resources: https://nda.nih.gov/study.html?id=3407 .\nKeep source data restrictions and teacher acknowledgements. Eligibility is not established by this release.\n\'\'\')\n    expected=set(files)|{\'transferred_labels.csv\',\'slot_audit.json\',\'original_study_labels_and_folds.csv\',\n        \'series_manifest.csv\',\'source_dataset_index.json\',\'preprocessing_config.json\',\'transfer_manifest.json\',\'README.md\'}\n    if {p.name for p in stage.iterdir()}!=expected: raise ValueError(\'Unexpected staging files\')\n    stage.rename(output)\n    return manifest\n\ndef validate_upload(output):\n    output=Path(output);m=json.loads((output/\'transfer_manifest.json\').read_text())\n    if digest(output/m[\'label_file\'])!=m[\'label_sha256\'] or digest(output/\'series_manifest.csv\')!=m[\'series_manifest_sha256\']:\n        raise ValueError(\'Upload table checksum mismatch\')\n    f=pd.read_csv(output/m[\'label_file\'],dtype={ID:str});s=pd.read_csv(output/\'series_manifest.csv\',dtype={ID:str,SID:str})\n    if len(f)!=m[\'studies\'] or not f[ID].is_unique or set(f[ID])!=set(s[ID]):raise ValueError(\'Upload coverage mismatch\')\n    for p in f[TARGETS].to_numpy():validate_probs(p)\n    for entry in m[\'image_archives\']:\n        p=output/entry[\'file\']\n        if p.is_symlink() or p.stat().st_size!=entry[\'bytes\'] or digest(p)!=entry[\'sha256\']:raise ValueError(\'Upload image checksum mismatch\')\n    return m\n', 'teacher_model.py': '"""Adapted from nartaa/rsna-knee-0949-anatomical-mirror v2 (Apache-2.0).\nOriginal: Danial Zakaria; Raptor architecture credited to dreaddevelopment.\nOnly model construction/loading retained; original execution code is not run.\n"""\nimport gc\nimport torch\nimport timm\nfrom torch import nn\nINPUT_RES = 320\ndef build_backbone(arch, pretrained=False):\n    # maxvit/maxxvit/coatnet are conv-attention hybrids: NO CLS token, NO interpolatable\n    # pos-embed -> avg pool. The "vit" substring in "coatnet"/"maxvit" must NOT route them\n    # down the ViT path (mirrors finetune_raptor.py exactly).\n    hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))\n    is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "dinov2", "eva", "beit"))\n    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3, img_size=INPUT_RES)\n    if is_vit:\n        kw.update(global_pool="token", dynamic_img_size=True)\n    else:\n        kw.update(global_pool="avg")\n    return timm.create_model(arch, **kw)\n\n\nclass RaptorClassifier(nn.Module):\n    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):\n        super().__init__()\n        self.backbone = backbone\n        self.norm = nn.LayerNorm(F_dim)\n        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop),\n                                 nn.Linear(256, n))\n        self.clsW = nn.Parameter(torch.zeros(n, F_dim))\n        self.clsb = nn.Parameter(torch.zeros(n))\n        nn.init.trunc_normal_(self.clsW, std=0.02)\n        self.n = n\n\n    def encode(self, x):\n        B, K = x.shape[:2]\n        # Bound backbone memory; the attention head still sees every window together.\n        flat = x.flatten(0, 1)\n        f = torch.cat([self.backbone(chunk) for chunk in flat.split(16)], dim=0)\n        return f.view(B, K, -1)\n\n    def head(self, feats):\n        h = self.norm(feats)\n        a = self.att(h)\n        a = torch.softmax(a, dim=1)\n        pooled = torch.einsum("bkn,bkf->bnf", a, h)\n        logits = (pooled * self.clsW).sum(-1) + self.clsb\n        return logits\n\n    def forward(self, x):\n        return self.head(self.encode(x))\n\n\ndef load_model(pt_path, arch_default, res_default, device, ngpu=1):\n    import hashlib\n    with open(pt_path, "rb") as stream:\n        observed_sha = hashlib.file_digest(stream, "sha256").hexdigest()\n    assert observed_sha == \'7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef\', "Bound checkpoint bytes differ"\n    print(f"[weights] sha256 {observed_sha}", flush=True)\n    ck = torch.load(pt_path, map_location="cpu", weights_only=False)\n    arch = ck.get("arch", arch_default)\n    assert int(ck.get("res", res_default)) == 384, "Expected original 384 checkpoint"\n    ck_res = INPUT_RES\n    bb = build_backbone(arch, pretrained=False)\n    model = RaptorClassifier(bb, F_dim=bb.num_features)\n    model.load_state_dict(ck["model"], strict=True)\n    model.eval().to(device)\n    # NOTE: DataParallel removed on purpose. On the full hidden test it drove a system-RAM OOM\n    # (per-forward module replication over many studies); a single T4 handles K_EVAL=24 windows\n    # fine. Arms are also run SEQUENTIALLY (see main) so peak RAM == one model, not two.\n    del ck\n    gc.collect()\n    return model, ck_res\n\n\n'}
CODE_DIR = Path('/kaggle/working/_coatnet_label_transfer_code')
CODE_DIR.mkdir(parents=True, exist_ok=True)
for name, source in SOURCE_FILES.items():
    (CODE_DIR / name).write_text(source)
sys.path.insert(0, str(CODE_DIR))
import runtime as transfer
importlib.reload(transfer)


<module 'runtime' from '/kaggle/working/_coatnet_label_transfer_code/runtime.py'>

## 2. Locate and validate inputs
Only active processed shards are accepted. Study/series IDs, geometry, masks, target schema, and the exact teacher checkpoint hash are checked. The source dataset is never modified.

In [4]:
import shutil
if not (NORMALIZED_ROOT / 'dataset_index.json').is_file():
    NORMALIZED_ROOT = Path(kagglehub.dataset_download(transfer.SOURCE))
if not (WEIGHTS_ROOT / transfer.WEIGHT).is_file():
    WEIGHTS_ROOT = Path(kagglehub.dataset_download(transfer.TEACHER))
WEIGHT_PATH = WEIGHTS_ROOT / transfer.WEIGHT
if transfer.digest(WEIGHT_PATH) != transfer.SHA: raise ValueError('Wrong teacher checkpoint')
if RESUME_ROOT is not None:
    if WORK_DIR.exists(): raise FileExistsError('Use an absent WORK_DIR when restoring mounted progress.')
    shutil.copytree(Path(RESUME_ROOT), WORK_DIR)
series, original_labels, source_index, preprocessing = transfer.inspect_source(NORMALIZED_ROOT)
print('Studies:', series.StudyInstanceUID.nunique(), '| usable series:', series.status.eq('ok').sum())
print('Excluded source series by status:', series.loc[~series.status.eq('ok'), 'status'].value_counts().to_dict())
print('Teacher SHA-256:', transfer.SHA)


Studies: 4407 | usable series: 24371
Excluded source series by status: {}
Teacher SHA-256: 7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef


In [5]:
from pathlib import Path
import hashlib
import torch

expected_weight = "raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt"
expected_sha256 = "7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef"

print("Normalized data:", NORMALIZED_ROOT)
print("Teacher weights:", WEIGHTS_ROOT)
print("GPU available:", torch.cuda.is_available())

assert Path(NORMALIZED_ROOT).is_dir(), "Normalized dataset path not found."
assert (Path(NORMALIZED_ROOT) / "dataset_index.json").is_file(), (
    "dataset_index.json not found. Check NORMALIZED_ROOT."
)
assert Path(WEIGHTS_ROOT).is_dir(), "Teacher weights dataset path not found."

matches = list(Path(WEIGHTS_ROOT).rglob(expected_weight))
assert len(matches) == 1, (
    f"Expected exactly one teacher checkpoint; found {len(matches)}: {matches}"
)

weight_path = matches[0]
print("Checkpoint:", weight_path)
print(f"Checkpoint size: {weight_path.stat().st_size / 1024**2:.1f} MB")
print("Verifying checkpoint checksum...", flush=True)

digest = hashlib.sha256()
with weight_path.open("rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        digest.update(chunk)

assert digest.hexdigest() == expected_sha256, (
    "Checkpoint checksum differs from the version expected by this notebook."
)
assert torch.cuda.is_available(), "Enable a GPU accelerator before running."

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

print("\nPASS: dataset index, teacher checkpoint, checksum, and GPU verified.")

Normalized data: /kaggle/input/datasets/gany24558/rsna-knee-normalized-all-training-data
Teacher weights: /kaggle/input/datasets/nartaa/rsna-knee-publication-swa-weights-20261007
GPU available: True
Checkpoint: /kaggle/input/datasets/nartaa/rsna-knee-publication-swa-weights-20261007/raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt
Checkpoint size: 279.3 MB
Verifying checkpoint checksum...
GPU 0: Tesla T4
GPU 1: Tesla T4

PASS: dataset index, teacher checkpoint, checksum, and GPU verified.


## 3. Generate transferred probabilities
Each completed study is saved immediately. An inference error stops execution; it never produces default 0.5 labels. Re-run to reuse completed predictions. If the time budget expires, save the notebook output, attach it to a new session, and set `RESUME_ROOT` to its `coatnet-label-transfer-progress` folder. Run Sections 1–3 again. Resume identity binds the source index, checkpoint and runtime code.


In [6]:
transferred, series, original_labels, source_index, preprocessing = transfer.run(
    NORMALIZED_ROOT, WEIGHT_PATH, WORK_DIR, max_hours=MAX_HOURS)
display(transferred.head())
display(transferred[transfer.TARGETS].describe().T)
print('Complete:', len(transferred), 'studies. These are teacher probabilities, not independent validation labels.')


[weights] sha256 7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef
Labelled 25 / 4407
Labelled 50 / 4407
Labelled 75 / 4407
Labelled 100 / 4407
Labelled 125 / 4407
Labelled 150 / 4407
Labelled 175 / 4407
Labelled 200 / 4407
Labelled 225 / 4407
Labelled 250 / 4407
Labelled 275 / 4407
Labelled 300 / 4407
Labelled 325 / 4407
Labelled 350 / 4407
Labelled 375 / 4407
Labelled 400 / 4407
Labelled 425 / 4407
Labelled 450 / 4407
Labelled 475 / 4407
Labelled 500 / 4407
Labelled 525 / 4407
Labelled 550 / 4407
Labelled 575 / 4407
Labelled 600 / 4407
Labelled 625 / 4407
Labelled 650 / 4407
Labelled 675 / 4407
Labelled 700 / 4407
Labelled 725 / 4407
Labelled 750 / 4407
Labelled 775 / 4407
Labelled 800 / 4407
Labelled 825 / 4407
Labelled 850 / 4407
Labelled 875 / 4407
Labelled 900 / 4407
Labelled 925 / 4407
Labelled 950 / 4407
Labelled 975 / 4407
Labelled 1000 / 4407
Labelled 1025 / 4407
Labelled 1050 / 4407
Labelled 1075 / 4407
Labelled 1100 / 4407
Labelled 1125 / 4407
Labelled 1150 /

,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,0.298743,0.551374,0.928258,0.450724,0.794318,0.357743,0.329088,0.775189,0.600208,0.706995,0.845368,0.900324
1,1.2.826.0.1.3680043.8.498.10004945927472656027...,0.266641,0.361415,0.101896,0.129931,0.174080,0.236305,0.304615,0.355209,0.335902,0.229182,0.805321,0.769135
2,1.2.826.0.1.3680043.8.498.10009278692606631573...,0.230355,0.276680,0.107603,0.112143,0.257648,0.218177,0.408822,0.188524,0.291595,0.175676,0.192222,0.444270
3,1.2.826.0.1.3680043.8.498.10009639203170750274...,0.268701,0.321253,0.241920,0.177248,0.186351,0.205876,0.217795,0.189775,0.290404,0.274985,0.281863,0.375301
4,1.2.826.0.1.3680043.8.498.10013663742400736029...,0.221931,0.278963,0.122272,0.195837,0.531867,0.563410,0.938211,0.434744,0.392829,0.304833,0.316337,0.400038


,count,mean,std,min,25%,50%,75%,max
ACL,4407.0,0.392721,0.225726,0.091622,0.234148,0.311788,0.470064,0.996789
MCL,4407.0,0.389167,0.153700,0.137586,0.277888,0.342508,0.456891,0.951384
Medial Meniscus,4407.0,0.458745,0.343508,0.035530,0.147893,0.306139,0.879862,0.986468
Lateral Meniscus,4407.0,0.314241,0.242612,0.032916,0.145716,0.224771,0.381241,0.990502
Medial OA,4407.0,0.395639,0.272010,0.032798,0.178516,0.290909,0.576363,0.992442
Lateral OA,4407.0,0.373465,0.218527,0.024550,0.208395,0.315239,0.496367,0.993921
PF OA,4407.0,0.413106,0.253667,0.032310,0.204441,0.337457,0.597409,0.990056
Effusion,4407.0,0.468429,0.272956,0.063936,0.213453,0.404279,0.740016,0.972406
Synovitis,4407.0,0.439200,0.156907,0.203169,0.304149,0.396346,0.553922,0.924408
Baker's,4407.0,0.406381,0.247743,0.110125,0.247367,0.294949,0.445335,0.994960


Complete: 4407 studies. These are teacher probabilities, not independent validation labels.


## 4. Package normalized images and labels
Requires space for a full copy of the source image archives plus a 2GB reserve. Copies are byte-checked. If interrupted while copying, rerun this cell to reuse complete copies. If the final folder already exists, go directly to Section 5.

The authoritative target file is **transferred_labels.csv**. The original label tables inside the copied archives remain original. The new dataset deliberately has `source_dataset_index.json`, rather than masquerading as an old training bundle: downstream training must explicitly join the teacher table by `StudyInstanceUID` and support soft-target BCE. All 12 probabilities are retained even for studies with original gold labels.


In [7]:
if OUTPUT_DIR.exists():
    package_info = transfer.validate_upload(OUTPUT_DIR)
else:
    package_info = transfer.package(NORMALIZED_ROOT, WORK_DIR, OUTPUT_DIR,
        transferred, series, original_labels, source_index, preprocessing, PUBLISH_HANDLE)
print('Ready:', OUTPUT_DIR)
print('Studies:', package_info['studies'], '| image archives:', len(package_info['image_archives']))


OSError: Not enough disk to copy image archives; keep WORK_DIR and retry with more space

## 5. Create the Kaggle dataset
Creates **gany24558/normalized-images-transferred-labels**, or a new version if it already exists, using KaggleHub. New datasets are private by default; existing visibility is unchanged. Keep MRI data private. Only the completed packaged folder is uploaded. Enable Internet for this cell.

After an upload failure, retain the output and retry this cell; a retry can create another version if a previous request succeeded. Upload submission is not confirmation that Kaggle has finished processing.


In [ ]:
package_info = transfer.validate_upload(OUTPUT_DIR)
if package_info['dataset_handle'] != PUBLISH_HANDLE: raise ValueError('Publication handle mismatch')
transfer.save_json(WORK_DIR / 'publication_status.json', {'status':'uploading','handle':PUBLISH_HANDLE})
try:
    kagglehub.dataset_upload(PUBLISH_HANDLE, str(OUTPUT_DIR),
        version_notes=f"CoAtNet soft-label transfer on normalized cache; {package_info['studies']} studies; image archives included.")
except Exception:
    transfer.save_json(WORK_DIR / 'publication_status.json', {'status':'failed_or_unconfirmed','handle':PUBLISH_HANDLE})
    raise
transfer.save_json(WORK_DIR / 'publication_status.json', {'status':'upload_submitted','handle':PUBLISH_HANDLE})
print('Dataset upload submitted: https://www.kaggle.com/datasets/' + PUBLISH_HANDLE)
print('Wait for Kaggle dataset processing to finish before using it for training.')
